# 🌿 AgriSense — MobileNetV2 Crop Disease Classifier

This notebook trains a **MobileNetV2** model on the PlantVillage dataset and exports the final weights to Google Drive as:
- `agrisense_mobilenetv2_finetuned.keras` — best fine-tuned model (primary)
- `agrisense_mobilenetv2.h5`             — phase-1 trained model (backup)
- `class_indices.json`                   — class-index mapping

**Steps:**
1. Mount Google Drive & set paths
2. Check GPU
3. Clone PlantVillage dataset from GitHub
4. Select 17 crop-disease classes (Corn, Tomato, Potato)
5. Split into train / val / test (70 / 15 / 15)
6. Build data pipelines (augmentation + normalisation)
7. Build MobileNetV2 model
8. **Phase 1** — Train classification head (base frozen)
9. **Phase 2** — Fine-tune top 30 layers of base
10. Evaluate on test set + confusion matrix

> **Runtime:** Set to *GPU → T4* for best speed. Training takes ~60-90 min total.

## Step 1 — Mount Google Drive & configure save paths

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# All model artefacts will be saved here
SAVE_DIR = '/content/drive/MyDrive/AgriSense/models'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_SAVE_PATH   = os.path.join(SAVE_DIR, 'agrisense_mobilenetv2.h5')
CLASSES_SAVE_PATH = os.path.join(SAVE_DIR, 'class_indices.json')

print('Drive mounted successfully!')
print('Target save directory:', SAVE_DIR)

## Step 2 — Verify GPU

In [ ]:
!nvidia-smi

## Step 3 — Clone PlantVillage dataset from GitHub

> The repo is ~2 GB; cloning takes a few minutes on Colab's fast connection.

In [ ]:
import os

!git clone https://github.com/spMohanty/PlantVillage-Dataset.git

SOURCE_DIR = '/content/PlantVillage-Dataset/raw/color'
classes = sorted(os.listdir(SOURCE_DIR))
print(f'Total classes available: {len(classes)}')
for c in classes:
    print(' ', c)

## Step 4 — Select target classes & copy to working directory

In [ ]:
import os
import json
import shutil

# 17 classes: Corn (4), Tomato (10), Potato (3)
selected_classes = [
    # Corn
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight',
    'Corn_(maize)___healthy',
    # Tomato
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy',
    # Potato
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
]

src_root = '/content/PlantVillage-Dataset/raw/color'
dst_root = '/content/agrisense_data'
os.makedirs(dst_root, exist_ok=True)

class_indices = {}
print('Copying selected crop-disease classes...')
for idx, cls in enumerate(sorted(selected_classes)):
    src = os.path.join(src_root, cls)
    dst = os.path.join(dst_root, cls)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        class_indices[str(idx)] = cls
        print(f'  Copied {cls}: {len(os.listdir(dst))} images')
    else:
        print(f'  MISSING: {cls}')

# Save class index map to Drive
with open(CLASSES_SAVE_PATH, 'w') as f:
    json.dump(class_indices, f, indent=4)

print(f'\nDataset ready: {len(class_indices)} classes')
print(f'Class map saved to: {CLASSES_SAVE_PATH}')

## Step 5 — Train / Val / Test split  (70 / 15 / 15)

In [ ]:
import os
import shutil
import random

random.seed(42)  # reproducible split

SPLIT_ROOT = '/content/agrisense_split'
splits = ['train', 'val', 'test']
ratios = [0.70, 0.15, 0.15]

for s in splits:
    os.makedirs(os.path.join(SPLIT_ROOT, s), exist_ok=True)

src_root = '/content/agrisense_data'
class_folders = sorted(os.listdir(src_root))

for cls in class_folders:
    cls_path = os.path.join(src_root, cls)
    images = os.listdir(cls_path)
    random.shuffle(images)

    n = len(images)
    n_train = int(n * ratios[0])
    n_val   = int(n * ratios[1])

    split_map = {
        'train': images[:n_train],
        'val':   images[n_train : n_train + n_val],
        'test':  images[n_train + n_val:],
    }

    for split_name, split_images in split_map.items():
        dst_dir = os.path.join(SPLIT_ROOT, split_name, cls)
        os.makedirs(dst_dir, exist_ok=True)
        for img in split_images:
            shutil.copy(
                os.path.join(cls_path, img),
                os.path.join(dst_dir, img)
            )

    print(f"{cls}: train={len(split_map['train'])}, val={len(split_map['val'])}, test={len(split_map['test'])}")

print('\nSplit complete. Data at:', SPLIT_ROOT)

## Step 6 — Build data pipelines (augmentation + normalisation)

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

train_dir = '/content/agrisense_split/train'
val_dir   = '/content/agrisense_split/val'
test_dir  = '/content/agrisense_split/test'

# Load raw datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=True, seed=42
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=False
)

class_names = train_ds.class_names
print('Class order:', class_names)
print('Number of classes:', len(class_names))

# Compute class weights to handle imbalance
print('\nComputing class weights...')
y_train = []
for _, labels in train_ds.unbatch():
    y_train.append(np.argmax(labels.numpy()))

classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
print(f'Weights computed for {len(class_weights)} classes.')

# Preprocessing layers
normalization_layer = tf.keras.layers.Rescaling(1./127.5, offset=-1)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
])

def prepare_train(ds):
    ds = ds.map(lambda x, y: (data_augmentation(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def prepare_eval(ds):
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = prepare_train(train_ds)
val_ds   = prepare_eval(val_ds)
test_ds  = prepare_eval(test_ds)

print('\nDatasets and preprocessing pipelines ready.')

## Step 7 — Build MobileNetV2 model

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

NUM_CLASSES = len(class_names)

# ImageNet-pretrained base — no classification head
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # frozen during phase 1

# Classification head
inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.3)(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

model.summary()

## Step 8 — Phase 1: Train classification head (base frozen)

Expected result: ~90% validation accuracy in 15 epochs.

In [ ]:
import os
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

EPOCHS = 15

callbacks = [
    ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights
)

print(f'\nPhase 1 complete! Best model saved to: {MODEL_SAVE_PATH}')

## Step 9 — Phase 2: Fine-tune top 30 base layers

> If the Colab session restarted, uncomment the two lines at the top to reload the model before fine-tuning.

In [ ]:
# Uncomment if session restarted:
# import tensorflow as tf
# model = tf.keras.models.load_model(MODEL_SAVE_PATH)

# Unfreeze base — only top 30 layers adapt to our diseases
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f'Total layers in base: {len(base_model.layers)}')
print(f'Unfrozen from layer {fine_tune_at} onward')

# Recompile with a much lower LR — avoids destroying phase-1 weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

FINETUNE_MODEL_PATH = os.path.join(SAVE_DIR, 'agrisense_mobilenetv2_finetuned.keras')
FINE_TUNE_EPOCHS    = 10

finetune_callbacks = [
    ModelCheckpoint(
        filepath=FINETUNE_MODEL_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=finetune_callbacks,
    class_weight=class_weights
)

print(f'\nFine-tuning complete! Best model saved to: {FINETUNE_MODEL_PATH}')

## Step 10 — Evaluate on test set + confusion matrix

In [ ]:
# Numeric metrics
test_results = model.evaluate(test_ds, verbose=1)

print('\n--- Final Test Set Results ---')
for name, value in zip(model.metrics_names, test_results):
    print(f'{name}: {value:.4f}')

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Per-class precision / recall / F1
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix — AgriSense CNN')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'))
plt.show()
print('Confusion matrix saved to Drive.')

## ✅ What to do after training

1. **Download from Drive** — go to `My Drive / AgriSense / models /`  
   and download `agrisense_mobilenetv2_finetuned.keras` (or the `.h5` backup).

2. **Rename and place** the file in your local project:
   ```
   ml/models/mobilenetv2_crop_disease.h5
   ```
   Also place `class_indices.json` in the same folder.

3. **Verify Tier-1 is active** (run from project root):
   ```bash
   python -c "from backend.app.services.disease_ml_service import DiseaseMLService; print('Tier:', DiseaseMLService()._tier)"
   ```
   Expected output: `Tier: 1`

4. **Run tests**:
   ```bash
   pytest tests
   ```